In [ ]:
!pip install -q vllm transformers loguru tqdm huggingface_hub dotenv

In [ ]:
from huggingface_hub import login
import dotenv
dotenv.load()
# Paste your HF token here (or set the HF_TOKEN env var in Colab Secrets)
HF_TOKEN = os.getenv("HF_TOKEN")

login(token=HF_TOKEN)
print("Logged in to HuggingFace ✓")

In [ ]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download

model_name = "mistralai/Devstral-Small-2-24B-Instruct-2512"

model_path = "/content/models/Devstral-Small-2-24B"

if Path(model_path).exists() and any(Path(model_path).iterdir()):
    print(f"Model already downloaded at {model_path} — skipping download ✓")
else:
    print(f"Downloading {model_name} → {model_path}")
    snapshot_download(
        repo_id=model_name,
        local_dir=model_path,
        token=HF_TOKEN,
        ignore_patterns=["*.md", "*.txt", "original/*"],  # skip docs/checkpoints
    )
    print(f"Saved to {model_path}")
    

In [ ]:
CONFIG = {
    "model_path": model_path,
    "gpus": 1,
    "max_num_seqs": 8,
    "gpu_memory_utilization": 0.95,
    "temperature": 0.0,
    "max_total_tokens": 8192,
    "max_new_tokens": 512,
    "repeat": 1,
    "testset_path": "./testsets.json",
    "output_path": "./output.json",
    "num_workers": 4,
}
os.environ["TOKENIZERS_PARALLELISM"] = "true"

print("Config loaded")


In [ ]:
import subprocess
import json
import traceback
import tempfile
import multiprocessing
import sys

from pathlib import Path
from tqdm import tqdm
from loguru import logger
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

logger.add(sys.stdout, colorize=False, format="{time} {level} {message}")

def evaluate_func(params):
    c_func = params["c_func"]
    c_test = params["c_test"]
    c_func_decompile = params["c_func_decompile"]
    
    timeout = 10
    flag_compile = 0
    flag_run = 0
    c_include = ""
    
    for line in c_func.split("\n"):
        if "#include" in line:
            c_include += line + "\n"
            c_func = c_func.replace(line, "")
    for line in c_test.split("\n"):
        if "#include" in line:
            c_include += line + "\n"
            c_test = c_test.replace(line, "")

    c_combine = c_include + "\n" + c_func_decompile + "\n" + c_test
    c_onlyfunc = c_include + "\n" + c_func_decompile

    with tempfile.TemporaryDirectory() as temp_dir:
        pid = os.getpid()
        c_file = os.path.join(temp_dir, f"combine_{pid}.c")
        executable = os.path.join(temp_dir, f"combine_{pid}")
        c_file_onlyfunc = os.path.join(temp_dir, f"onlyfunc_{pid}.c")
        executable_onlyfunc = os.path.join(temp_dir, f"onlyfunc_{pid}")

        with open(c_file, "w") as f:
            f.write(c_combine)
        with open(c_file_onlyfunc, "w") as f:
            f.write(c_onlyfunc)
        
        try:
            subprocess.run(
                ["gcc", "-S", c_file_onlyfunc, "-o", executable_onlyfunc, "-lm"],
                check=True,
                timeout=timeout,
            )
            flag_compile = 1
        except:
            return flag_compile, flag_run
        
        try:
            subprocess.run(
                ["gcc", c_file, "-o", executable, "-lm"]
                check=True, timeout=timeout, capture_output=True,
            )
        except Exception:
            return flag_compile, flag_run
        
        try:
            subprocess.run(
                [executable], capture_output=True, text=True, timeout=timeout, check=True
            )
            flag_compile = 1
        except Exception:
            return flag_compile, flag_run'
        
        try: 
            subprocess.run(
                [executable], capture_output=True, text=True,
                timeout=timeout, check=True
            )
            flag_run = 1
        except Exception:
            return flag_compile, flag_run
    
    return flag_compile, flag_run


In [ ]:
def decompile_pass_rate(testsets, gen_results_repeat, opts, num_workers):
    all_stats = []

    for gen_results in gen_results_repeat:
        ctx = multiprocessing.get_context("spawn")
        tasks = [
            {
                "c_func": testset["c_func"],
                "c_test": testset["c_test"],
                "c_func_decompile": output[0],
            }
            for testset, output in zip(testsets,gen_results)
        ]

        with ctx.Pool(num_workers) as pool:
            eval_results = list(tqdm(pool.imap(evaluate_func, tasks), total=len(tasks)))
        
        stats = {opt: {"compile": 0, "run": 0, "total": 0} for opt in opts}
        for testset, output, flag in tqdm(
            zip(testsets, gen_results, eval_results),
            total=len(testsets),
            desc="Tallying",
        ):
            opt = testset["type"]
            flag_compile, flag_run = flag
            stats[opt]["total"] += 1
            if flag_compile:
                stats[opt]["compile"] += 1
            if flag_run:
                stats[opt]["run"] += 1
        all_stats.append(stats)
    
    avg_stats = {opt: {"compile":0, "run": 0, "total": 0} for opt in opts}
    for stats in all_stats:
        for opt in opts:
            avg_stats[opt]["compile"] += stats[opt]["compile"]
            avg_stats[opt]["run"] += stats[opt]["run"]
            avg_stats[opt]["total"] += stats[opt]["total"]
        for opt, data in avg_stats.items():
            compile_rate = data["compile"] / data["total"] if data["total"] > 0 else 0
            run_rate = data["run"] / data["total"] if data["total"] > 0 else 0
            print(f"  Opt {opt}: Compile {compile_rate:.2%}  |  Run {run_rate:.2%}")

    return avg_stats
print("Functions defined ✓")


In [ ]:
def run_eval_pipeline(cfg: dict):
    testset_path = cfg["testset_path"]
    output_path = cfg["output_path"]

    if not Path(testset_path).exists():
        raise ValueError(f"Testset path {testset_path} does not exist")
    if not Path(output_path).exists():
        raise ValueError(f"Output path {output_path} does not exist")

    with open(testset_path, "r") as f:
        testsets = json.load(f)
    logger.info(f"Loaded testset with {len(testsets)} cases")

    opts = {
        "O0": "# This is the assembly code:\n",
        "O1": "# This is the assembly code:\n",
        "O2": "# This is the assembly code:\n",
        "O3": "# This is the assembly code:\n",
    }
    after = "\n# What is the source code?\n"

    inputs = []
    for testset in testsets:
        opt = testset["type"]
        prompt = opts[opt] + testset["input_asm_prompt"] + after
        inputs.append(prompt)
    
    tokenizer = AutoTokenizer.from_pretrained(cfg["model_path"])
    stop_sequences = [tokenizer.eos_token]

    logger.info(f"loading model {cfg['model_path']}")
    llm = LLM(
        model=cfg["model_path"],
        tensor_parallel_size=cfg["gpus"],
        max_model_len=cfg["max_total_tokens"],
        gpu_memory_utilization=cfg["gpu_memory_utilization"],
    )

    sampling_params = SamplingParams(
        temperature=cfg["temperature"],
        max_tokens=cfg["max_new_tokens"],
        stop=stop_sequences,
    )

    gen_results_repeat = []
    logger.info(f"Loading model: {cfg['model_path']}")
    llm = LLM(
        model=cfg["model_path"],
        tensor_parallel_size=cfg["gpus"],
        max_model_len=cfg["max_total_tokens"],
        gpu_memory_utilization=cfg["gpu_memory_utilization"],
    )
    sampling_params = SamplingParams(
        temperature=cfg["temperature"],
        max_tokens=cfg["max_new_tokens"],
        stop=stop_sequences,
    )

    gen_results_repeat = []
    logger.info(f"Running {cfg['repeat']} generation pass(es)")
    for i in range(cfg["repeat"]):
        logger.info(f"Generation pass {i+1}/{cfg['repeat']}")
        raw = llm.generate(inputs, sampling_params)
        gen_results = [[out.outputs[0].text] for out in raw]
        gen_results_repeat.append(gen_results)
    
    if output_path:
        save_data = []
        for testset, res in zip(testsets, gen_results_repeat):
            testset["output"] = res[0]
            save_data.append(testset)
        with open(output_path, "w") as f:
            json.dump(save_data, f, indent=4, ensure_ascii=True)
        logger.info(f"Saved predictions to {output_path}")
    
    avg_stats = decompile_pass_rate(
        testsets,
        gen_results_repeat,
        opts,
        cfg["num_workers"]
    )
    return avg_stats

In [ ]:
results = run_eval_pipeline(cfg)